# Merge Predicted Basic Attributes

Attach the lightweight model's predicted `length` and `curl` outputs to the reviewed asset bank as auxiliary metadata. These predictions are hints, not replacements for reviewed labels.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

PROJECT_ROOT

WindowsPath('.')

In [2]:
import json
import pandas as pd

from systems.static_auto_tryon.auto_app.ml.celeba_hair_rich import merge_basic_predictions_into_assets
from systems.static_auto_tryon.auto_app.ml.datasets import read_jsonl_manifest, write_jsonl_manifest

REVIEWED_ROOT = BACKEND_ROOT / 'data' / 'processed' / 'celeba_hair_rich_assets' / 'reviewed'
PREDICTION_ROOT = BACKEND_ROOT / 'data' / 'processed' / 'celeba_hair_rich_assets' / 'predictions'

LABELED_ASSETS_JSONL = REVIEWED_ROOT / 'kept_assets_labeled.jsonl'
PREDICTIONS_JSONL = PREDICTION_ROOT / 'predicted_basic_attributes.jsonl'

ENRICHED_JSONL = REVIEWED_ROOT / 'kept_assets_labeled_enriched.jsonl'
ENRICHED_CSV = REVIEWED_ROOT / 'kept_assets_labeled_enriched.csv'
ENRICHED_SUMMARY_JSON = REVIEWED_ROOT / 'kept_assets_labeled_enriched_summary.json'

REVIEWED_ROOT

WindowsPath('./backend/data/processed/celeba_hair_rich_assets/reviewed')

In [3]:
asset_rows = read_jsonl_manifest(LABELED_ASSETS_JSONL)
prediction_rows = read_jsonl_manifest(PREDICTIONS_JSONL)

print('Labeled assets:', len(asset_rows))
print('Prediction rows:', len(prediction_rows))
asset_rows[0] if asset_rows else None

Labeled assets: 120
Prediction rows: 250


{'asset_id': 'celeba_hair_000002',
 'source_dataset': 'CelebA',
 'original_celeba_file': '000006.jpg',
 'raw_image_path': 'backend/data\\raw\\celeba\\img_align_celeba\\img_align_celeba\\000006.jpg',
 'raw_mask_path': 'backend/data\\datasets\\celeba_hair_rich\\segmentation_masks\\000006_hair_mask.png',
 'image_path': 'backend/data\\processed\\celeba_hair_rich_assets\\images\\celeba_hair_000002.png',
 'mask_path': 'backend/data\\processed\\celeba_hair_rich_assets\\masks\\celeba_hair_000002_mask.png',
 'partition': 'train',
 'gender_label': 'female',
 'quality_score': 1.0,
 'confidence_bucket': 'high',
 'quality_flags': ['usable_candidate'],
 'hair_mask_stats': {'positive_pixels': 15657,
  'coverage_ratio': 0.403489,
  'bbox_x': 7,
  'bbox_y': 17,
  'bbox_width': 171,
  'bbox_height': 201},
 'crop_box': {'left': 0, 'top': 0, 'right': 178, 'bottom': 218},
 'crop_size': {'width': 178, 'height': 218},
 'celeba_attribute_hints': {'positive_fields': ['Brown_Hair', 'Wavy_Hair'],
  'color_hint':

In [4]:
enriched_rows, missing_predictions = merge_basic_predictions_into_assets(asset_rows, prediction_rows)

print('Enriched rows:', len(enriched_rows))
print('Missing predictions:', len(missing_predictions))
missing_predictions[:10]

Enriched rows: 120
Missing predictions: 0


[]

In [5]:
write_jsonl_manifest(enriched_rows, ENRICHED_JSONL)

flat_rows = []
for row in enriched_rows:
    normalized = row['labeling']['normalized_attributes']
    predictions = row.get('predicted_basic_attributes', {})
    flat_rows.append({
        'asset_id': row['asset_id'],
        'gender_label': row.get('gender_label'),
        'quality_score': row.get('quality_score'),
        'review_length': normalized.get('length'),
        'review_curl': normalized.get('curl'),
        'pred_length': predictions.get('length', {}).get('label'),
        'pred_length_confidence': predictions.get('length', {}).get('confidence'),
        'pred_curl': predictions.get('curl', {}).get('label'),
        'pred_curl_confidence': predictions.get('curl', {}).get('confidence'),
    })

enriched_frame = pd.DataFrame(flat_rows)
enriched_frame.to_csv(ENRICHED_CSV, index=False, encoding='utf-8')

summary = {
    'total_enriched_assets': len(enriched_rows),
    'missing_predictions': len(missing_predictions),
    'review_length_counts': enriched_frame['review_length'].fillna('').replace('', '(blank)').value_counts().to_dict(),
    'review_curl_counts': enriched_frame['review_curl'].fillna('').replace('', '(blank)').value_counts().to_dict(),
    'pred_length_counts': enriched_frame['pred_length'].fillna('').replace('', '(blank)').value_counts().to_dict(),
    'pred_curl_counts': enriched_frame['pred_curl'].fillna('').replace('', '(blank)').value_counts().to_dict(),
    'avg_pred_length_confidence': round(float(enriched_frame['pred_length_confidence'].dropna().mean()), 4),
    'avg_pred_curl_confidence': round(float(enriched_frame['pred_curl_confidence'].dropna().mean()), 4),
}
ENRICHED_SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Wrote:', ENRICHED_JSONL)
print('Wrote:', ENRICHED_CSV)
print('Wrote:', ENRICHED_SUMMARY_JSON)

Wrote: backend/data\processed\celeba_hair_rich_assets\reviewed\kept_assets_labeled_enriched.jsonl
Wrote: backend/data\processed\celeba_hair_rich_assets\reviewed\kept_assets_labeled_enriched.csv
Wrote: backend/data\processed\celeba_hair_rich_assets\reviewed\kept_assets_labeled_enriched_summary.json


In [6]:
enriched_frame.head(20)

,asset_id,gender_label,quality_score,review_length,review_curl,pred_length,pred_length_confidence,pred_curl,pred_curl_confidence
0,celeba_hair_000002,female,1.00,long,wavy,short,0.650244,straight,0.736030
1,celeba_hair_000003,male,0.96,short,straight,short,0.687539,straight,0.683670
2,celeba_hair_000004,male,0.96,short,NaN,short,0.467601,straight,0.512496
3,celeba_hair_000005,female,1.00,medium,NaN,short,0.475137,straight,0.491533
4,celeba_hair_000006,male,0.96,medium,straight,short,0.548526,straight,0.502465
5,celeba_hair_000009,male,1.00,short,straight,short,0.704176,straight,0.640301
6,celeba_hair_000011,female,1.00,long,wavy,short,0.782798,straight,0.717937
7,celeba_hair_000012,female,1.00,long,wavy,short,0.597149,straight,0.751261
8,celeba_hair_000013,male,0.96,long,NaN,short,0.517352,wavy,0.510737
9,celeba_hair_000014,female,1.00,long,wavy,short,0.724770,straight,0.565562


In [7]:
pd.Series(summary)

total_enriched_assets                                                       120
missing_predictions                                                           0
review_length_counts          {'long': 57, 'short': 51, 'medium': 9, '(blank...
review_curl_counts            {'straight': 45, 'wavy': 39, '(blank)': 34, 'c...
pred_length_counts                                   {'short': 110, 'long': 10}
pred_curl_counts                                  {'straight': 101, 'wavy': 19}
avg_pred_length_confidence                                               0.6599
avg_pred_curl_confidence                                                 0.6397
dtype: object